In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# === DeBERTa-v3-base for Extractive QA (SQuAD v2) ===
# Run in Kaggle notebook (GPU recommended).

# 1) Installs
!pip install -q transformers datasets evaluate accelerate

# 2) Imports
import numpy as np
import random
from datasets import load_dataset, load_metric
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
)
import evaluate

# 3) Config
MODEL_NAME = "microsoft/deberta-v3-base"
MAX_LENGTH = 512
DOC_STRIDE = 128
BATCH_SIZE = 8
NUM_EPOCHS = 3
OUTPUT_DIR = "/kaggle/working/deberta_qa"
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

# 4) Load dataset
squad = load_dataset("squad_v2")
train_val = squad["train"].train_test_split(test_size=0.20, seed=SEED)
train_ds = train_val["train"]
val_ds = train_val["test"]
test_ds = squad["validation"]

print("train:", len(train_ds), "val:", len(val_ds), "test:", len(test_ds))

# 5) Tokenizer & model
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)

# 6) Preprocessing for QA (creates features with token mapping)
def prepare_train_features(examples):
    # examples: dict with 'question', 'context', 'answers'
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )

    # Since one example might give several features due to long contexts, we need to map features -> example
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized_examples.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        input_ids = tokenized_examples["input_ids"][i]
        cls_index = input_ids.index(tokenizer.cls_token_id) if tokenizer.cls_token_id in input_ids else 0

        sequence_ids = tokenized_examples.sequence_ids(i)

        sample_index = sample_mapping[i]
        answers = examples["answers"][sample_index]
        if len(answers["answer_start"]) == 0:
            # Unanswerable
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            # take first answer (SQuAD has multiple answers)
            start_char = answers["answer_start"][0]
            end_char = start_char + len(answers["text"][0])

            # Find token start and end within the feature
            token_start_index = 0
            while sequence_ids[token_start_index] != 1:
                token_start_index += 1
            token_end_index = len(input_ids) - 1
            while sequence_ids[token_end_index] != 1:
                token_end_index -= 1

            # If answer is outside the span
            if not (offsets[token_start_index][0] <= start_char and offsets[token_end_index][1] >= end_char):
                start_positions.append(cls_index)
                end_positions.append(cls_index)
            else:
                # otherwise, find the exact token indices
                token_index = token_start_index
                while token_index <= token_end_index and offsets[token_index][0] <= start_char:
                    token_index += 1
                start_positions.append(token_index - 1)

                token_index = token_end_index
                while token_index >= token_start_index and offsets[token_index][1] >= end_char:
                    token_index -= 1
                end_positions.append(token_index + 1)

    tokenized_examples["start_positions"] = start_positions
    tokenized_examples["end_positions"] = end_positions
    return tokenized_examples

def prepare_validation_features(examples):
    # For validation we need offset mappings to convert prediction tokens back to chars
    tokenized_examples = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length"
    )
    sample_mapping = tokenized_examples.pop("overflow_to_sample_mapping")
    tokenized_examples["example_id"] = []

    for i in range(len(tokenized_examples["input_ids"])):
        sample_index = sample_mapping[i]
        tokenized_examples["example_id"].append(examples["id"][sample_index])
        # keep offset mapping for answer extraction; set to None for question tokens
        sequence_ids = tokenized_examples.sequence_ids(i)
        offsets = tokenized_examples["offset_mapping"][i]
        tokenized_examples["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None for k, o in enumerate(offsets)
        ]
    return tokenized_examples

# Map datasets
train_features = train_ds.map(prepare_train_features, batched=True, remove_columns=train_ds.column_names)
val_features = val_ds.map(prepare_validation_features, batched=True, remove_columns=val_ds.column_names)
test_features = test_ds.map(prepare_validation_features, batched=True, remove_columns=test_ds.column_names)

# 7) Metrics helper: use 'squad_v2' metric from evaluate
squad_metric = evaluate.load("squad_v2")

# We'll need a post-processing function to get final answers from predictions
from collections import OrderedDict
import collections
def postprocess_qa_predictions(examples, features, raw_predictions, n_best_size=20, max_answer_length=30):
    # raw_predictions: tuple (start_logits, end_logits)
    all_start_logits, all_end_logits = raw_predictions
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, f in enumerate(features):
        features_per_example[f["example_id"]].append(i)

    predictions = OrderedDict()

    for example in examples:
        example_id = example["id"]
        context = example["context"]
        feature_indices = features_per_example[example_id]

        min_null_score = None
        valid_answers = []

        for fi in feature_indices:
            start_logits = all_start_logits[fi]
            end_logits = all_end_logits[fi]
            offset_mapping = features[fi]["offset_mapping"]

            cls_index = features[fi]["input_ids"].index(tokenizer.cls_token_id) if tokenizer.cls_token_id in features[fi]["input_ids"] else 0
            # Score of null (no answer)
            null_score = start_logits[cls_index] + end_logits[cls_index]
            if min_null_score is None or null_score < min_null_score:
                min_null_score = null_score

            # top n_best_size start/end
            start_indexes = np.argsort(start_logits)[-1: -n_best_size - 1 : -1].tolist()
            end_indexes = np.argsort(end_logits)[-1: -n_best_size - 1 : -1].tolist()

            for start_index in start_indexes:
                for end_index in end_indexes:
                    # Skip invalid combinations
                    if start_index >= len(offset_mapping) or end_index >= len(offset_mapping):
                        continue
                    if offset_mapping[start_index] is None or offset_mapping[end_index] is None:
                        continue
                    if end_index < start_index or end_index - start_index + 1 > max_answer_length:
                        continue

                    start_char = offset_mapping[start_index][0]
                    end_char = offset_mapping[end_index][1]
                    answer_text = context[start_char:end_char]
                    score = start_logits[start_index] + end_logits[end_index]
                    valid_answers.append({"score": score, "text": answer_text})

        if len(valid_answers) > 0:
            best_answer = sorted(valid_answers, key=lambda x: x["score"], reverse=True)[0]
        else:
            best_answer = {"text": "", "score": 0.0}

        # SQuAD v2: compare best non-null to null_score and choose empty string if null is better
        if min_null_score is not None and min_null_score > best_answer["score"]:
            predictions[example_id] = ""
        else:
            predictions[example_id] = best_answer["text"]

    return predictions

# 8) Trainer setup and training
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    logging_steps=200,
    fp16=True,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

def compute_metrics_for_qa(eval_pred):
    # eval_pred: (start_logits, end_logits, features)
    start_logits, end_logits = eval_pred.predictions
    # Map back to examples
    predictions = postprocess_qa_predictions(
        examples=val_ds,  # original validation examples
        features=val_features,
        raw_predictions=(start_logits, end_logits)
    )
    references = {ex["id"]: ex["answers"] for ex in val_ds}
    # Format predictions & references for metric
    formatted_predictions = [{"id": k, "prediction_text": v} for k, v in predictions.items()]
    formatted_references = [{"id": ex["id"], "answers": ex["answers"]} for ex in val_ds]
    results = squad_metric.compute(predictions=formatted_predictions, references=formatted_references)
    # squad_metric returns {'exact': x, 'f1': y, 'total': N, 'HasAns_exact':..., 'HasAns_f1':..., ...}
    return {"exact": results["exact"], "f1": results["f1"]}

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_features,
    eval_dataset=val_features,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics_for_qa
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# 9) Final evaluation on test set
raw_pred = trainer.predict(test_features)
start_logits, end_logits = raw_pred.predictions
predicted_answers = postprocess_qa_predictions(
    examples=test_ds,
    features=test_features,
    raw_predictions=(start_logits, end_logits)
)

# Compute official SQuAD v2 metric
formatted_predictions = [{"id": k, "prediction_text": v} for k, v in predicted_answers.items()]
formatted_references = [{"id": ex["id"], "answers": ex["answers"]} for ex in test_ds]
results = evaluate.load("squad_v2").compute(predictions=formatted_predictions, references=formatted_references)
print("Test SQuAD v2 results:", results)

# 10) Example inference for your pipeline: given context paragraph(s) and generated questions, extract answers
def extract_answer_for(context, question, max_length=MAX_LENGTH, stride=DOC_STRIDE):
    # Tokenize and run single example (handles long contexts by sliding window)
    inputs = tokenizer(question, context, truncation="only_second", max_length=max_length, stride=stride, return_overflowing_tokens=True, return_offsets_mapping=True, padding="max_length", return_tensors="pt")
    input_ids = inputs["input_ids"]
    offset_mapping = inputs["offset_mapping"]
    example_id = inputs.get("overflow_to_sample_mapping", [0]*input_ids.shape[0])
    model.to("cuda" if next(model.parameters()).is_cuda else "cpu")
    with torch.no_grad():
        outputs = model(input_ids.to(model.device))
    start_logits = outputs.start_logits.cpu().numpy()
    end_logits = outputs.end_logits.cpu().numpy()
    # Use the same postprocess logic but for a single example
    # Simplest: pick best start+end across windows with valid offsets
    best_answer = ""
    best_score = -1e9
    for i in range(input_ids.shape[0]):
        offsets = offset_mapping[i].tolist()
        for start_idx in np.argsort(start_logits[i])[-1:-20:-1]:
            for end_idx in np.argsort(end_logits[i])[-1:-20:-1]:
                if offsets[start_idx][0] is None or offsets[end_idx][1] is None:
                    continue
                if end_idx < start_idx:
                    continue
                score = start_logits[i][start_idx] + end_logits[i][end_idx]
                if score > best_score:
                    start_char = offsets[start_idx][0]
                    end_char = offsets[end_idx][1]
                    candidate = context[start_char:end_char]
                    best_answer = candidate
                    best_score = score
    return best_answer

# Example:
# context = test_ds[0]["context"]
# question = "What is ...?"  # maybe from Flan-T5 predictions
# print(extract_answer_for(context, question))